# Kaggle: Document Corners Training (MobileNetV3-Large 0.75, 2xT4)

Standalone notebook for corner regression only.
- No UNet/segmentation branch
- DDP training on 2 GPUs (`ddp_notebook`)
- Saves checkpoints, TensorBoard logs, and `results.csv`


In [1]:
# Optional: install only missing package for MobileNetV3-Large 0.75 weights
# Kaggle already has most dependencies preinstalled; avoid reinstalling pandas/jupyter.
!pip install -q --no-deps timm==1.0.15


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.0 MB/s eta 0:00:00


In [2]:
import ast
import random
import re
from pathlib import Path
from typing import Dict, List, Optional

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
from albumentations.pytorch import ToTensorV2
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.regression import MeanAbsoluteError

try:
    import timm
except ImportError:
    timm = None


In [3]:
WORKING_ROOT = Path('/kaggle/working')
IMAGES_DIR = "/kaggle/input/datasets/uznikpostmoderna/documentcorners/images"
ANNOTATIONS_PATH = "/kaggle/input/datasets/uznikpostmoderna/documentcorners/annotations.csv"
OUTPUT_DIR = WORKING_ROOT / 'corner_training_output'
PRETRAINED_WEIGHTS_PATH = '/kaggle/input/models/timm/tf-mobilenet-v3/pytorch/tf-mobilenetv3-large-075/1/tf_mobilenetv3_large_075-150ee8b0.pth'

BATCH_SIZE = 64
MAX_EPOCHS = 40
IMG_SIZE = 640
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 2025
NUM_WORKERS = 2

SPLIT_TRAIN = 0.7
SPLIT_VAL = 0.15
SPLIT_TEST = 0.15

CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
TB_DIR = OUTPUT_DIR / 'tb_logs'
RESULTS_PATH = OUTPUT_DIR / 'results.csv'


In [4]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    pl.seed_everything(seed, workers=True)


# Accept both str and Path values from config cell
WORKING_ROOT = Path(WORKING_ROOT)
IMAGES_DIR = Path(IMAGES_DIR)
ANNOTATIONS_PATH = Path(ANNOTATIONS_PATH)
OUTPUT_DIR = Path(OUTPUT_DIR)
PRETRAINED_WEIGHTS_PATH = Path(PRETRAINED_WEIGHTS_PATH)
CHECKPOINT_DIR = Path(CHECKPOINT_DIR)
TB_DIR = Path(TB_DIR)
RESULTS_PATH = Path(RESULTS_PATH)

if not PRETRAINED_WEIGHTS_PATH.exists():
    raise FileNotFoundError(f'Pretrained weights not found: {PRETRAINED_WEIGHTS_PATH}')
if not ANNOTATIONS_PATH.exists():
    raise FileNotFoundError(f'annotations.csv not found: {ANNOTATIONS_PATH}')
if not IMAGES_DIR.exists():
    raise FileNotFoundError(f'images directory not found: {IMAGES_DIR}')

gpu_count = torch.cuda.device_count()
print(f'GPU count: {gpu_count}')
if gpu_count < 2:
    raise RuntimeError('This notebook requires 2 GPUs (T4) for training.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
TB_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)


Seed set to 2025


GPU count: 2


In [5]:
LINE_PATTERN = re.compile(
    r"^(?P<filename>[^,]+),(?P<filepath>[^,]+),(?P<width>[^,]+),(?P<height>[^,]+),(?P<coords>\[\[.*?\]\]),(?P<visibility>\[.*?\]),(?P<dataset>[^,]+)$"
)


def order_points_tl_tr_br_bl(points_xy: np.ndarray) -> np.ndarray:
    pts = np.asarray(points_xy, dtype=np.float32)
    if pts.shape != (4, 2):
        raise ValueError(f'Expected (4,2), got {pts.shape}')

    pts_sorted_y = pts[np.argsort(pts[:, 1])]
    top = pts_sorted_y[:2]
    bottom = pts_sorted_y[2:]

    top = top[np.argsort(top[:, 0])]
    bottom = bottom[np.argsort(bottom[:, 0])]

    tl, tr = top
    bl, br = bottom
    return np.stack([tl, tr, br, bl], axis=0)


def parse_annotation_line(line: str) -> Optional[Dict]:
    match = LINE_PATTERN.match(line.strip())
    if match is None:
        return None

    payload = match.groupdict()
    try:
        coords = np.asarray(ast.literal_eval(payload['coords']), dtype=np.float32)
        _ = ast.literal_eval(payload['visibility'])
    except (ValueError, SyntaxError):
        return None

    payload['coords'] = coords
    return payload


def resolve_image_path(raw_filepath: str, filename: str, annotations_path: Path, images_root: Path) -> Optional[Path]:
    candidate = Path(raw_filepath)
    if candidate.is_absolute():
        primary = candidate
    else:
        primary = annotations_path.parent / candidate

    if primary.exists():
        return primary.resolve()

    fallback = images_root / filename
    if fallback.exists():
        return fallback.resolve()

    return None


def load_corner_records(annotations_path: Path, images_root: Path) -> tuple[list[dict], dict]:
    records = []
    stats = {
        'total': 0,
        'kept': 0,
        'dropped_parse': 0,
        'dropped_coords': 0,
        'dropped_image': 0,
    }

    with annotations_path.open('r', encoding='utf-8') as src:
        _ = src.readline()
        for line in src:
            if not line.strip():
                continue

            stats['total'] += 1
            parsed = parse_annotation_line(line)
            if parsed is None:
                stats['dropped_parse'] += 1
                continue

            coords = parsed['coords']
            if coords.shape != (4, 2) or (not np.isfinite(coords).all()) or (coords < 0.0).any() or (coords > 1.0).any():
                stats['dropped_coords'] += 1
                continue

            coords = order_points_tl_tr_br_bl(coords)

            image_path = resolve_image_path(parsed['filepath'], parsed['filename'], annotations_path, images_root)
            if image_path is None:
                stats['dropped_image'] += 1
                continue

            records.append({
                'filename': parsed['filename'],
                'filepath': str(image_path),
                'keypoints_norm': coords.astype(np.float32),
            })
            stats['kept'] += 1

    return records, stats


def split_records(records: List[Dict], seed: int) -> Dict[str, List[Dict]]:
    frac_sum = SPLIT_TRAIN + SPLIT_VAL + SPLIT_TEST
    if abs(frac_sum - 1.0) > 1e-6:
        raise ValueError(f'Split fractions must sum to 1.0, got {frac_sum}')

    indices = np.arange(len(records))
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)

    n_total = len(records)
    n_train = int(n_total * SPLIT_TRAIN)
    n_val = int(n_total * SPLIT_VAL)

    train_idx = indices[:n_train]
    val_idx = indices[n_train:n_train + n_val]
    test_idx = indices[n_train + n_val:]

    return {
        'train': [records[i] for i in train_idx],
        'val': [records[i] for i in val_idx],
        'test': [records[i] for i in test_idx],
    }


In [6]:
records, stats = load_corner_records(ANNOTATIONS_PATH, IMAGES_DIR)
if not records:
    raise RuntimeError('No valid records after filtering.')

splits = split_records(records, SEED)

print('Annotation stats:', stats)
print('Split sizes:', {k: len(v) for k, v in splits.items()})


Annotation stats: {'total': 4492, 'kept': 3890, 'dropped_parse': 0, 'dropped_coords': 602, 'dropped_image': 0}
Split sizes: {'train': 2723, 'val': 583, 'test': 584}


In [7]:
def get_train_transforms(size: int):
    return A.Compose(
        [
            A.Resize(height=size, width=size, interpolation=cv2.INTER_LINEAR),
            A.RandomBrightnessContrast(p=0.3),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.GaussNoise(p=0.3),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0),
            ToTensorV2(),
        ],
        keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
    )


def get_valid_transforms(size: int):
    return A.Compose(
        [
            A.Resize(height=size, width=size, interpolation=cv2.INTER_LINEAR),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0),
            ToTensorV2(),
        ],
        keypoint_params=A.KeypointParams(format='xy', remove_invisible=False),
    )


class CornersDataset(Dataset):
    def __init__(self, records: List[Dict], transforms, with_targets: bool = True):
        self.records = records
        self.transforms = transforms
        self.with_targets = with_targets

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int):
        rec = self.records[idx]
        image_path = rec['filepath']
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if image is None:
            raise ValueError(f'Image does not exist: {image_path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        if self.with_targets:
            keypoints_norm = np.asarray(rec['keypoints_norm'], dtype=np.float32).reshape(4, 2)
            scale = np.asarray([max(w - 1, 1), max(h - 1, 1)], dtype=np.float32)
            keypoints_px = keypoints_norm * scale

            transformed = self.transforms(image=image, keypoints=keypoints_px.tolist())
            image_t = transformed['image'].float()
            kps_t = np.asarray(transformed['keypoints'], dtype=np.float32).reshape(4, 2)
            kps_t = order_points_tl_tr_br_bl(kps_t)

            h2, w2 = int(image_t.shape[1]), int(image_t.shape[2])
            scale2 = np.asarray([max(w2 - 1, 1), max(h2 - 1, 1)], dtype=np.float32)
            kps_norm_t = np.clip(kps_t / scale2, 0.0, 1.0)

            target = torch.from_numpy(kps_norm_t.reshape(-1))
            return image_t, rec['filename'], rec['filepath'], target

        transformed = self.transforms(image=image, keypoints=[])
        image_t = transformed['image'].float()
        return image_t, rec['filename'], rec['filepath']


def collate_train(batch):
    images, filenames, filepaths, targets = zip(*batch)
    return torch.stack(images), list(filenames), list(filepaths), torch.stack(targets)


def collate_infer(batch):
    images, filenames, filepaths = zip(*batch)
    return torch.stack(images), list(filenames), list(filepaths)


class CornersDataModule(pl.LightningDataModule):
    def __init__(self, splits: Dict[str, List[Dict]]):
        super().__init__()
        self.splits = splits

    def setup(self, stage: Optional[str] = None):
        if stage in (None, 'fit'):
            self.train_dataset = CornersDataset(self.splits['train'], get_train_transforms(IMG_SIZE), True)
            self.val_dataset = CornersDataset(self.splits['val'], get_valid_transforms(IMG_SIZE), True)
        if stage in (None, 'predict', 'infer'):
            self.predict_dataset = CornersDataset(self.splits['test'], get_valid_transforms(IMG_SIZE), False)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            collate_fn=collate_train,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            collate_fn=collate_train,
        )

    def predict_dataloader(self):
        return DataLoader(
            self.predict_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            collate_fn=collate_infer,
        )


In [8]:
class MobileNetV3Corners(nn.Module):
    def __init__(self, width_mult: float = 0.75, pretrained_weights_path: Optional[Path] = None, num_keypoints: int = 4):
        super().__init__()
        if abs(width_mult - 0.75) > 1e-8:
            raise ValueError('This notebook is fixed to width_mult=0.75 for mobilenetv3_large_075.')

        if timm is None:
            raise ImportError('timm is required for MobileNetV3-Large 0.75. Run pip install timm.')

        self.backbone = timm.create_model(
            'mobilenetv3_large_075',
            pretrained=False,
            num_classes=num_keypoints * 2,
        )

        if pretrained_weights_path is not None:
            self.load_pretrained_weights(pretrained_weights_path)

    def load_pretrained_weights(self, weights_path: Path) -> None:
        checkpoint = torch.load(weights_path, map_location='cpu')

        if isinstance(checkpoint, dict):
            if 'state_dict' in checkpoint and isinstance(checkpoint['state_dict'], dict):
                state_dict = checkpoint['state_dict']
            elif 'model' in checkpoint and isinstance(checkpoint['model'], dict):
                state_dict = checkpoint['model']
            elif 'model_state_dict' in checkpoint and isinstance(checkpoint['model_state_dict'], dict):
                state_dict = checkpoint['model_state_dict']
            else:
                state_dict = checkpoint
        else:
            raise TypeError(f'Unsupported checkpoint format: {type(checkpoint)}')

        cleaned_state_dict = {}
        for key, value in state_dict.items():
            new_key = key
            if new_key.startswith('module.'):
                new_key = new_key[len('module.'):]
            if new_key.startswith('backbone.'):
                new_key = new_key[len('backbone.'):]
            cleaned_state_dict[new_key] = value

        model_state = self.backbone.state_dict()
        filtered_state_dict = {}
        skipped_shape = []

        for key, value in cleaned_state_dict.items():
            if key not in model_state:
                continue
            if model_state[key].shape != value.shape:
                skipped_shape.append(key)
                continue
            filtered_state_dict[key] = value

        if not filtered_state_dict:
            raise RuntimeError(f'No compatible keys found in pretrained weights: {weights_path}')

        missing, unexpected = self.backbone.load_state_dict(filtered_state_dict, strict=False)
        missing_non_classifier = [k for k in missing if not k.startswith('classifier.')]
        if missing_non_classifier:
            raise RuntimeError(f'Missing non-classifier keys after loading weights: {missing_non_classifier[:10]}')

        print(
            f'Loaded pretrained weights from {weights_path}. '
            f'Loaded keys: {len(filtered_state_dict)}, '
            f'Skipped (shape mismatch): {len(skipped_shape)}, '
            f'Unexpected: {len(unexpected)}'
        )

    def forward(self, x):
        return self.backbone(x)


class CornersLightningModule(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = MobileNetV3Corners(
            width_mult=0.75,
            pretrained_weights_path=PRETRAINED_WEIGHTS_PATH,
            num_keypoints=4,
        )
        self.loss_fn = nn.SmoothL1Loss(beta=0.05)
        self.val_mae = MeanAbsoluteError()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=5,
            threshold=1e-4,
            threshold_mode='rel',
        )
        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'monitor': 'val_mae',
                'interval': 'epoch',
                'frequency': 1,
            },
        }

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, _, _, targets = batch
        preds = torch.sigmoid(self(images))
        loss = self.loss_fn(preds, targets)
        mae = torch.mean(torch.abs(preds - targets))
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log('train_mae', mae, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, _, _, targets = batch
        preds = torch.sigmoid(self(images))
        loss = self.loss_fn(preds, targets)
        self.val_mae.update(preds, targets)
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)

    def on_validation_epoch_start(self):
        self.val_mae.reset()

    def on_validation_epoch_end(self):
        self.log('val_mae', self.val_mae.compute(), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)

    def predict_step(self, batch, batch_idx):
        images, filenames, filepaths = batch
        preds = torch.sigmoid(self(images))
        return preds, filenames, filepaths


In [9]:
datamodule = CornersDataModule(splits)
model = CornersLightningModule()

checkpoint_callback = ModelCheckpoint(
    dirpath=str(CHECKPOINT_DIR),
    monitor='val_mae',
    mode='min',
    filename='epoch_{epoch:02d}-{val_mae:.5f}',
    save_last=True,
    verbose=True,
)

logger = TensorBoardLogger(save_dir=str(TB_DIR), name='mobilenetv3_075')

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=2,
    strategy='ddp_notebook',
    precision='16-mixed',
    callbacks=[
        checkpoint_callback,
        EarlyStopping(monitor='val_mae', mode='min', patience=10),
        LearningRateMonitor(logging_interval='epoch'),
    ],
    logger=logger,
    log_every_n_steps=20,
)

trainer.fit(model=model, datamodule=datamodule)

best_ckpt = CHECKPOINT_DIR / 'best_corners.ckpt'
last_ckpt = CHECKPOINT_DIR / 'last.ckpt'

if checkpoint_callback.best_model_path:
    torch.save(torch.load(checkpoint_callback.best_model_path, map_location='cpu'), best_ckpt)
else:
    raise RuntimeError('No best checkpoint path after training.')

if checkpoint_callback.last_model_path:
    torch.save(torch.load(checkpoint_callback.last_model_path, map_location='cpu'), last_ckpt)

print('Best checkpoint:', best_ckpt)
print('Last checkpoint:', last_ckpt)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Loaded pretrained weights from /kaggle/input/models/timm/tf-mobilenet-v3/pytorch/tf-mobilenetv3-large-075/1/tf_mobilenetv3_large_075-150ee8b0.pth. Loaded keys: 310, Skipped (shape mismatch): 2, Unexpected: 0


Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 2 processes
----------------------------------------------------------------------------------------------------

2026-04-23 21:04:03.217012: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776978243.572493      66 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776978243.677963      66 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776978244.516372      66

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ MobileNetV3Corners │  2.7 M │ train │     0 │
│ 1 │ loss_fn │ SmoothL1Loss       │      0 │ train │     0 │
│ 2 │ val_mae │ MeanAbsoluteError  │      0 │ train │     0 │
└───┴─────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 2.7 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.7 M                                                                                                
Total estimated model params size (MB): 10                                                                         
Modules in train mode: 299                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad node created prior to the current iteration being kept alive. This can happen if the autograd graph is still being kept alive by tensors such as the loss, or if you are using DDP, which will stash a reference to the node. To resolve the mismatch, delete all references to the autograd graph or ensure that DDP initialization is performed under the same stream as subsequent forwards. If the mismatch is intentional, you can use torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False) to suppress this warning. (Triggered internally at /pytorch/torch/csrc/autograd/input_buffer.cpp:240.)
  return Variable._execution_en

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 18. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 36. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

Epoch 0, global step 22: 'val_mae' reached 0.17031 (best 0.17031), saving model to '/kaggle/working/corner_training_output/checkpoints/epoch_epoch=00-val_mae=0.17031.ckpt' as top 1
Epoch 1, global step 44: 'val_mae' reached 0.13415 (best 0.13415), saving model to '/kaggle/working/corner_training_output/checkpoints/epoch_epoch=01-val_mae=0.13415.ckpt' as top 1
Epoch 2, global step 66: 'val_mae' reached 0.09866 (best 0.09866), saving model to '/kaggle/working/corner_training_output/checkpoints/epoch_epoch=02-val_mae=0.09866.ckpt' as top 1
Epoch 3, global step 88: 'val_mae' reached 0.06539 (best 0.06539), saving model to '/kaggle/working/corner_training_output/checkpoints/epoch_epoch=03-val_mae=0.06539.ckpt' as top 1
Epoch 4, global step 110: 'val_mae' reached 0.04953 (best 0.04953), saving model to '/kaggle/working/corner_training_output/checkpoints/epoch_epoch=04-val_mae=0.04953.ckpt' as top 1
Epoch 5, global step 132: 'val_mae' reached 0.04458 (best 0.04458), saving model to '/kaggle/w

Best checkpoint: /kaggle/working/corner_training_output/checkpoints/best_corners.ckpt
Last checkpoint: /kaggle/working/corner_training_output/checkpoints/last.ckpt


In [10]:
# Run prediction with a single-device trainer to get Python-side outputs.
predict_trainer = pl.Trainer(
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    logger=False,
    enable_checkpointing=False,
)

pred_batches = predict_trainer.predict(
    model=model,
    datamodule=datamodule,
    ckpt_path=str(best_ckpt),
    return_predictions=True,
)

if pred_batches is None:
    raise RuntimeError(
        'predict() returned None. Use a single-device prediction trainer with distributed training strategies.'
    )

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at /kaggle/working/corner_training_output/checkpoints/best_corners.ckpt
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/trainer/call.py:283: Be aware that when using `ckpt_path`, callbacks used to create the checkpoint need to be provided during `Trainer` instantiation. Please add the following callbacks: ["EarlyStopping{'monitor': 'val_mae', 'mode': 'min'}", "ModelCheckpoint{'monitor': 'val_mae', 'mode': 'min', 'every_n_train_steps': 0, 'every_n_epochs': 1, 'train_time_interval': None}"].
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /kaggle/working/corner_training_output/checkpoints/best

Output()

In [11]:
print('Prediction batches:', len(pred_batches))

Prediction batches: 10


In [12]:
# Inference on test split and save results.csv
rows = []
for preds, filenames, filepaths in pred_batches:
    preds_np = preds.detach().cpu().numpy()
    for i, kps in enumerate(preds_np):
        row = {
            'filename': filenames[i],
            'filepath': filepaths[i],
        }
        for kp_idx in range(4):
            row[f'x{kp_idx + 1}'] = float(kps[2 * kp_idx])
            row[f'y{kp_idx + 1}'] = float(kps[2 * kp_idx + 1])
        rows.append(row)

results_df = pd.DataFrame(rows)
results_df.to_csv(RESULTS_PATH, index=False)

print('Saved results to:', RESULTS_PATH)
results_df.head()

Saved results to: /kaggle/working/corner_training_output/results.csv


,filename,filepath,x1,y1,x2,y2,x3,y3,x4,y4
0,1835.jpg,/kaggle/input/datasets/uznikpostmoderna/docume...,0.019170,0.526938,0.346658,0.214819,0.885038,0.655582,0.491959,0.934578
1,1468.jpg,/kaggle/input/datasets/uznikpostmoderna/docume...,0.093994,0.171289,0.807072,0.209405,0.737707,0.890943,0.061179,0.852933
2,0254.jpg,/kaggle/input/datasets/uznikpostmoderna/docume...,0.121055,0.123335,0.729581,0.128592,0.615557,0.728180,0.062876,0.767606
3,2847.jpg,/kaggle/input/datasets/uznikpostmoderna/docume...,0.250332,0.065797,0.924607,0.036794,0.946199,0.726032,0.280963,0.733572
4,1035.jpg,/kaggle/input/datasets/uznikpostmoderna/docume...,0.118860,0.240927,0.747539,0.243213,0.745728,0.895961,0.128113,0.876801


In [13]:
print('Artifacts:')
print('-', CHECKPOINT_DIR / 'best_corners.ckpt')
print('-', CHECKPOINT_DIR / 'last.ckpt')
print('-', RESULTS_PATH)
print('-', TB_DIR)


Artifacts:
- /kaggle/working/corner_training_output/checkpoints/best_corners.ckpt
- /kaggle/working/corner_training_output/checkpoints/last.ckpt
- /kaggle/working/corner_training_output/results.csv
- /kaggle/working/corner_training_output/tb_logs


<a href="/kaggle/working/corner_training_output/tb_logs"> Download File </a>
